# Validierung Wohnlagen 2026
Separates Notebook f?r Abgleiche gegen Ortsteile, Quartiere und Mietkategorien.


In [ ]:
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
from shapely.geometry import box
from scipy.stats import chi2_contingency

from helper import load_geocsv

CITY_BOUNDING_BOX = box(12.3120236786, 52.2938432979, 12.7562682548, 52.5594777244)
gdf = load_geocsv("out/wohnlagen_brb_2026.csv").to_crs(4326)
gdf_quartiere = gpd.read_file("data/Quartiere/2024_Quartiere.gpkg").to_crs(4326)
gdf_ortsteile = gpd.read_file("data/ortsteile_brandenburg.json", bbox=CITY_BOUNDING_BOX).to_crs(4326)

gdf = gdf.drop(columns=["index_right", "quartier", "mietkategorie", "ortsteil", "bezeichnun", "mietkatego", "otl_name"], errors="ignore")
gdf = gpd.sjoin(gdf, gdf_quartiere[["bezeichnun", "mietkatego", "geometry"]], how="left", predicate="within")
gdf = gdf.drop(columns=["index_right"], errors="ignore")
gdf = gpd.sjoin(gdf, gdf_ortsteile[["otl_name", "geometry"]], how="left", predicate="within")
gdf = gdf.drop(columns=["index_right"], errors="ignore")
gdf = gdf.rename(columns={"bezeichnun": "quartier", "mietkatego": "mietkategorie", "otl_name": "ortsteil"})


In [ ]:
for col, normalize in [
    ("quartier", "index"),
    ("ortsteil", "index"),
    ("mietkategorie", "columns"),
]:
    ct = pd.crosstab(gdf[col], gdf["cluster_skater_block"], normalize=normalize)
    display(ct.round(2))
    plt.figure(figsize=(10, max(4, len(ct) * 0.3)))
    sns.heatmap(ct, cmap="Blues", annot=False)
    plt.title(f"SKATER-Cluster nach {col}")
    plt.tight_layout()
    plt.show()
